# Link Prediction in Social Networks
## Notebook 03: Supervised Machine Learning, Deep Learning & Benchmarking

### Objectives
1. Train baseline heuristic predictors, Logistic Regression, Random Forest, XGBoost, LightGBM, MLP, and Spectral GCN.
2. Evaluate models using ROC-AUC, PR-AUC, F1-Score, Precision, Recall, and Hits@10%.
3. Plot ROC/PR curves and analyze feature importances.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

from src.data_loader import load_graph, split_edges_leak_free
from src.pipeline import LinkPredictionPipeline
from src.models import train_model_zoo
from src.evaluator import evaluate_all_models, get_feature_importances

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

### 1. Load Data and Extract Features

In [ ]:
G = load_graph(dataset_name="facebook_ego", sample_nodes=800, data_dir="../data/raw")
splits = split_edges_leak_free(G, test_ratio=0.15, val_ratio=0.05, seed=42)
pipeline = LinkPredictionPipeline(use_embeddings=True, embedding_dim=16, seed=42)
data_dict = pipeline.fit_transform_splits(splits)

### 2. Train All Models

In [ ]:
models = train_model_zoo(data_dict)

### 3. Evaluate and Display Leaderboard

In [ ]:
leaderboard_df, detailed_metrics = evaluate_all_models(models, data_dict)
leaderboard_df

### 4. Plot Receiver Operating Characteristic (ROC) & Precision-Recall (PR) Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
ax1.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC=0.50)')
for name, m in detailed_metrics.items():
    if "Heuristic (c" not in name and "Heuristic (p" not in name:  # Plot best models
        ax1.plot(m['fpr'], m['tpr'], label=f"{name} (AUC={m['roc_auc']:.3f})")
ax1.set_title("ROC Curves Comparison")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend(loc="lower right")

# PR Curve
for name, m in detailed_metrics.items():
    if "Heuristic (c" not in name and "Heuristic (p" not in name:
        ax2.plot(m['recall_curve'], m['precision_curve'], label=f"{name} (PR-AUC={m['pr_auc']:.3f})")
ax2.set_title("Precision-Recall Curves Comparison")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

### 5. Feature Importance Analysis (XGBoost)

In [ ]:
fi_df = get_feature_importances(models["XGBoost"], data_dict["feature_names"]).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=fi_df, palette="Blues_r")
plt.title("Top 10 Most Predictive Features for Link Formation (XGBoost)")
plt.xlabel("Feature Importance Score")
plt.ylabel("Feature Name")
plt.tight_layout()
plt.show()